# Analytics Agent — Agentic Analytics & Pattern Discovery
### Business Analytics Demo

---

## What is the Analytics Agent?

The **Analytics Agent** is the agentic counterpart of Level-2 Analytics. It receives natural-language requests, routes them through the **Orchestrator → Analytics Agent → MCP Analytics Server** pipeline, and returns insights enriched with:

- **Agent routing** — which agents were invoked and why
- **MCP tool calls** — which server tools were used
- **Memory** — observations recorded for future learning

| What you can ask | What the agent does |
|---|---|
| "Segment customers by risk and engagement" | Routes to analytics_agent → MCP generate_segment |
| "Top customers by balance" | Routes to analytics_agent → MCP run_sql_query |
| "Analyze high fraud risk customers" | Routes to analytics_agent → MCP run_sql_query |
| "Analyze sentiment from customer feedback" | Direct NLP + memory observation |
| "Show Customer 360 for CUST-005" | Routes to analytics_agent → SQL query |

---

## How it works (Agentic vs Nonagent)

```
Nonagent:  Your question → Level-2 Agent → Direct tool call → Response

Agentic:   Your question
                │
                ▼
         Orchestrator Agent  ← classifies intent
                │
                ▼
         Analytics Agent     ← selects MCP tool
                │
                ▼
         MCP Analytics Server (run_sql_query / generate_segment)
                │
                ▼
         Memory Layer        ← records observation
                │
                ▼
         Response + Agent Path + MCP Calls
```

Every analysis is logged to the audit trail and the agent records observations to memory for future learning.

---

## Setup

Run this cell once to initialise the agentic system.

In [1]:
import sys, os
from pathlib import Path
from dotenv import load_dotenv

_root = Path(os.getcwd())
while not (_root / "nonagentic" / "__init__.py").exists() and _root != _root.parent:
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "notebooks"))
load_dotenv()
os.environ["GUARDRAIL_ENABLED"] = "false"

import json
import pandas as pd
from IPython.display import display, HTML

from notebooks.agentic.utils import (
    agentic_ask,
    agentic_ask_dynamic,
    show_agentic_result,
    show_agentic_extras,
    record_nlp_observation,
    show_agent_plan,
    show_agent_reasoning,
    show_tool_selection,
    show_sql_generated,
    verify_outputs_match,
)
from notebooks.nonagentic.utils.demo_data import get_demo_customers, get_test_queries
from notebooks.nonagentic.utils.visualization import (
    display_segmentation_results,
    display_sql_analytics_results,
    display_fraud_risk_analysis,
    display_enhanced_sentiment_analysis,
    display_text_summarization,
    display_customer_360_view,
    display_sales_analytics_dashboard,
    display_support_analytics_dashboard,
)

print("✅ Agentic Analytics Ready")
print(f"📁 Project root: {_root}")


✅ Agentic Analytics Ready
📁 Project root: /home/kahloka/projects/agenticAI


---
## Use Case 1 — Customer Segmentation (K-means Clustering)

**Business scenario**: The marketing team wants to identify distinct customer groups based on risk and engagement scores to tailor campaigns.

**This use case demonstrates TWO execution modes:**
- **Version A (Deterministic)**: Traditional fixed routing through analytics agent
- **Version B (Dynamic)**: Planner generates execution plan, discovers tools dynamically

Both versions produce identical results, but Version B provides full observability of the decision-making process.

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# Version A: Deterministic Execution (Original Behavior)
# ═══════════════════════════════════════════════════════════════════════════════

result_deterministic = agentic_ask("Segment customers by risk and engagement scores into 4 groups")

# Show routing + result
show_agentic_result(result_deterministic, title="Customer Segmentation (Deterministic)")

# Display segments using shared visualization
final = result_deterministic.get("final_result") or {}
segments = final.get("segments", [])
if segments:
    display_segmentation_results(segments)

Segment,Size,Avg Risk Score,Avg Engagement
casual,31,0.234,0.444
dormant_vip,53,0.255,0.246
at_risk,39,0.246,0.335
vip,53,0.126,0.739
new,32,0.350,0.251


### Version B: Dynamic Agentic Execution

Now let's see the same request executed with **dynamic planning and tool discovery**:
1. **Planner** analyzes the query and generates an execution plan
2. **Tool Registry** discovers available tools dynamically
3. **Agent** selects the best tool and executes
4. **Memory** records observations for learning

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# Version B: Dynamic Agentic Execution (New Behavior)
# ═══════════════════════════════════════════════════════════════════════════════

from agentic.planner import generate_plan
from agentic.registry import tool_registry

query = "Segment customers by risk and engagement scores into 4 groups"

# Step 1: Generate execution plan
plan = generate_plan(query)
show_agent_plan(plan)
show_agent_reasoning(plan.get('reasoning', ''), plan.get('analysis')) # type: ignore

# Step 2: Discover available tools
tools = tool_registry.find_tools_by_capability("segment")
show_tool_selection(tools, selected="generate_segment")

# Step 3: Execute with dynamic planning
result_dynamic = agentic_ask_dynamic(query)
show_agentic_result(result_dynamic, title="Customer Segmentation (Dynamic)")

# Display segments (same visualization)
final = result_dynamic.get("final_result") or {}
segments = final.get("segments", [])
if segments:
    display_segmentation_results(segments)

Segment,Size,Avg Risk Score,Avg Engagement
casual,31,0.234,0.444
dormant_vip,53,0.255,0.246
at_risk,39,0.246,0.335
vip,53,0.126,0.739
new,32,0.350,0.251


In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# Verification: Both Versions Produce Identical Results
# ═══════════════════════════════════════════════════════════════════════════════

verify_outputs_match(result_deterministic, result_dynamic)

True

In [6]:
# Show agentic extras (agent path, MCP calls, memory)
show_agentic_extras(result_dynamic)

---
## Use Case 2 — SQL Analytics (Top Customers by Balance)

**Business scenario**: The account manager needs to identify top customers by lifetime value for VIP treatment.

**This use case demonstrates:**
- **Version A (Deterministic)**: Direct SQL execution through analytics agent
- **Version B (Dynamic)**: SQL generated from natural language, validated for safety

Version B showcases the **SQL Generator** that converts natural language to safe SQL queries.

In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# Version A: Deterministic Execution (Original Behavior)
# ═══════════════════════════════════════════════════════════════════════════════

queries = get_test_queries()
result_deterministic = agentic_ask(
    f"Run this SQL query: {queries['top_customers_sql']}",
)

show_agentic_result(result_deterministic, title="Top Customers by Lifetime Value (Deterministic)")

final = result_deterministic.get("final_result") or {}
display_sql_analytics_results(final, title="💰 Top 10 Customers by Lifetime Value")

Customer ID,Name,Segment,Lifetime Value,Fraud Score,Engagement
CUST-004,Sharon Kennedy,at_risk,"€89,205.38",0.300,0.440
CUST-104,الأستاذ ثروت خندف,dormant_vip,"€88,661.71",0.570,0.200
CUST-149,Michelle Wallace,dormant_vip,"€88,343.24",0.570,0.240
CUST-037,صدّام جرهم,new,"€87,845.00",0.220,0.590
CUST-165,Gary Morales,vip,"€87,761.18",0.300,0.900
CUST-045,Lukácsné Kovács Beát,new,"€87,456.71",0.870,0.680
CUST-073,Sheri Johnson,dormant_vip,"€87,217.77",0.060,0.070
CUST-067,Dr. Kiss Ildikó,casual,"€86,056.98",0.050,0.570
CUST-131,لاما عكاوي,dormant_vip,"€86,006.77",0.720,0.180
CUST-048,Veres Mária,new,"€85,543.75",0.590,0.430


### Version B: Dynamic SQL Generation

Now let's generate the SQL from natural language:
1. **SQL Generator** converts "top customers by lifetime value" to SQL
2. **Validator** ensures the query is safe (SELECT only)
3. **Optimizer** adds LIMIT and formats properly
4. **Explainer** generates human-readable description

In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# Version B: Dynamic SQL Generation (New Behavior)
# ═══════════════════════════════════════════════════════════════════════════════

from agentic.sql import generate_sql, validate_sql, explain_sql
from agentic.planner import generate_plan

nl_query = "Top 10 customers by lifetime value"

# Step 1: Generate plan
plan = generate_plan(f"Show me {nl_query}")
show_agent_plan(plan)

# Step 2: Generate SQL from natural language
sql = generate_sql(nl_query)
is_valid, errors = validate_sql(sql)
explanation = explain_sql(sql)

show_sql_generated(sql, explanation)

if not is_valid:
    print(f"❌ SQL validation failed: {errors}")
else:
    # Step 3: Execute
    result_dynamic = agentic_ask(f"Run this SQL query: {sql}")
    show_agentic_result(result_dynamic, title="Top Customers (Dynamic SQL)")
    
    final = result_dynamic.get("final_result") or {}
    display_sql_analytics_results(final, title="💰 Top 10 Customers by Lifetime Value")

Customer ID,Name,Segment,Lifetime Value,Fraud Score,Engagement
CUST-004,Sharon Kennedy,at_risk,"€89,205.38",0.300,0.440
CUST-104,الأستاذ ثروت خندف,dormant_vip,"€88,661.71",0.570,0.200
CUST-149,Michelle Wallace,dormant_vip,"€88,343.24",0.570,0.240
CUST-037,صدّام جرهم,new,"€87,845.00",0.220,0.590
CUST-165,Gary Morales,vip,"€87,761.18",0.300,0.900
CUST-045,Lukácsné Kovács Beát,new,"€87,456.71",0.870,0.680
CUST-073,Sheri Johnson,dormant_vip,"€87,217.77",0.060,0.070
CUST-067,Dr. Kiss Ildikó,casual,"€86,056.98",0.050,0.570
CUST-131,لاما عكاوي,dormant_vip,"€86,006.77",0.720,0.180
CUST-048,Veres Mária,new,"€85,543.75",0.590,0.430


In [9]:
# Verify outputs match
verify_outputs_match(result_deterministic, result_dynamic)

True

In [10]:
show_agentic_extras(result_dynamic)

---
## Use Case 3 — Fraud Risk Analysis (High-Fraud-Risk Customers)

**Business scenario**: Trust & Safety needs to identify high-fraud-risk customers for enhanced verification.

**Agentic flow**: Orchestrator → Analytics Agent → MCP `run_sql_query` with fraud filter SQL.

The agent autonomously constructs the SQL, executes it read-only, and records a fraud-risk observation to memory.

In [11]:
fraud_sql = """
SELECT segment, COUNT(*) as count, AVG(fraud_score) as avg_risk, AVG(engagement_score) as avg_engagement
FROM customers
WHERE fraud_score > 0.7
GROUP BY segment
ORDER BY count DESC
"""

result = agentic_ask(f"Run this SQL query: {fraud_sql}")

show_agentic_result(result, title="Fraud Risk Analysis")

final = result.get("final_result") or {}
display_fraud_risk_analysis(final)


segment,count,avg_risk,avg_engagement
at_risk,17,0.838,0.448
dormant_vip,16,0.814,0.143
casual,11,0.822,0.494
new,10,0.797,0.491
vip,9,0.890,0.821


In [12]:
show_agentic_extras(result)


---
## Use Case 4 — Sentiment Analysis (Social Media Feedback)

**Business scenario**: The brand team wants to understand customer sentiment from social media posts.

**Agentic pattern**: NLP tools (RoBERTa) are called directly — the analytics agent does not wrap NLP in MCP. After analysis, the result is recorded to **agent_observation memory** so the system learns about brand sentiment trends.

**RoBERTa** (`cardiffnlp/twitter-roberta-base-sentiment-latest`) classifies each post as positive, neutral, or negative.

In [13]:
from nonagentic.tools.nlp import analyze_sentiment

with open("data/social_media.json") as f:
    social_data = json.load(f)

sample_posts = social_data[:10]
results = []
for post in sample_posts:
    sentiment_result = analyze_sentiment(post["text"])
    results.append({
        "Text": post["text"][:60] + "..." if len(post["text"]) > 60 else post["text"],
        "Platform": post["platform"],
        "Sentiment": sentiment_result.get("sentiment", "unknown"),
        "Confidence": f"{sentiment_result.get('confidence', 0):.1%}"
    })

display_enhanced_sentiment_analysis(results, sample_posts=sample_posts)

# Record to agentic memory
sentiments = [r["Sentiment"] for r in results]
dominant = max(set(sentiments), key=sentiments.count)
record_nlp_observation("Social media sentiment", f"dominant={dominant} across {len(results)} posts")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text,Platform,Sentiment,Confidence
Been a loyal customer for years. Never disappointed!,facebook,positive,96.7%
Return process is unnecessarily complicated. Avoid!,twitter,negative,84.5%
Poor quality for the price. Expected much better,instagram,negative,91.2%
Lightning fast response from customer service team,facebook,positive,68.6%
Website keeps crashing during checkout. Lost my cart multipl...,facebook,negative,92.3%
Product categories are well organized on the site,reddit,positive,84.1%
Lightning fast response from customer service team,facebook,positive,68.6%
They have a good variety of brands available,instagram,positive,88.7%
Do they ship internationally? Need to send a gift abroad,instagram,neutral,87.0%
Smooth checkout process and arrived perfectly packaged,instagram,positive,95.4%


### Agentic Extras — Memory Observation

The sentiment result is recorded to `agent_observation` memory so the system can track brand sentiment trends over time.

In [14]:
from memory.memory_manager import memory_manager
from notebooks.nonagentic.utils.display_helpers import display_metrics

observations = memory_manager.agent_observation.get_observations("analytics")
display_metrics({
    "Observations recorded": len(observations),
    "Latest": observations[-1]["observation"] if observations else "(none)",
})


---
## Use Case 5 — Text Summarization (Support Tickets)

**Business scenario**: Support managers need quick summaries of lengthy support tickets.

**Agentic pattern**: BART summarization is called directly (not via MCP), then the summary is recorded to agent_observation memory.

**BART** (`facebook/bart-large-cnn`) generates concise summaries with `compression_ratio`.

In [15]:
from nonagentic.tools.nlp import summarize_text

with open("data/support_tickets.json") as f:
    support_data = json.load(f)

sample_ticket = support_data[0]
display_text_summarization(sample_ticket, summarize_text)

# Record to agentic memory
record_nlp_observation("Support ticket summarized", f"ticket_id={sample_ticket.get('id', 'unknown')}")


In [16]:
observations = memory_manager.agent_observation.get_observations("analytics")
display_metrics({
    "Observations recorded": len(observations),
    "Latest": observations[-1]["observation"] if observations else "(none)",
})


---
## Use Case 6 — Customer 360 View (Unified Data)

**Business scenario**: An account manager needs a complete view of a customer before a meeting.

**Agentic flow**: `agentic_ask` → Orchestrator → Analytics Agent → MCP `run_sql_query` to pull customer data, then the agent aggregates CRM, sales, social, and support data into a unified view.

Data from **CRM, sales, social media, and support** is aggregated into one unified view with summary metrics.

In [17]:
from nonagentic.tools.customer360 import get_customer_360

demo_customers = get_demo_customers()
customer_id = demo_customers["customer_360_demo"]

# Customer 360 is a direct data aggregation (not analytics routing)
# Unlike UC1-3 which use agentic_ask → analytics_agent → MCP,
# Customer 360 directly aggregates data from multiple sources
customer_360 = get_customer_360(customer_id)

# Display the unified view
display_customer_360_view(customer_360, customer_id)


Category
home
books
clothing


Channel,Consent
marketing,True
email,True
sms,True
phone,False


In [ ]:
# Note: UC6 uses direct tool call (not agentic routing)
# so there are no MCP calls or agent path to display


---
## Use Case 7 — Sales Analytics (Revenue by Category)

**Business scenario**: The sales team wants to understand which categories generate the most revenue.

**Agentic flow**: Orchestrator → Analytics Agent → MCP `run_sql_query` to aggregate sales data by product category and channel.

Returns `total_revenue`, `by_product`, `by_channel`, and `top_product`.

In [18]:
from nonagentic.tools.customer360 import get_sales_analytics

result = agentic_ask("Analyze sales analytics by product category and channel")
show_agentic_result(result, title="Sales Analytics")

# Fetch full sales data and display dashboard
sales_analytics = get_sales_analytics()
display_sales_analytics_dashboard(sales_analytics)


In [19]:
show_agentic_extras(result)


---
## Use Case 8 — Support Analytics (Ticket Analysis)

**Business scenario**: The support manager needs to understand ticket volume, types, and resolution times.

**Agentic flow**: Orchestrator → Analytics Agent → MCP `run_sql_query` to aggregate support ticket data.

Returns `avg_resolution_hours` and `high_priority` count.

In [20]:
from nonagentic.tools.customer360 import get_support_analytics

result = agentic_ask("Analyze support ticket volume, types, and resolution times")
show_agentic_result(result, title="Support Analytics")

# Fetch full support data and display dashboard
support_analytics = get_support_analytics()
display_support_analytics_dashboard(support_analytics)


In [21]:
show_agentic_extras(result)


---
## Summary

| Use Case | What the agent does | Agentic Layer |
|---|---|---|
| **1. Customer Segmentation** | K-means clustering via MCP generate_segment | analytics_agent → MCP |
| **2. SQL Analytics** | Top customers via MCP run_sql_query | analytics_agent → MCP |
| **3. Fraud Risk Analysis** | High-fraud filter via MCP run_sql_query | analytics_agent → MCP |
| **4. Sentiment Analysis** | RoBERTa classification + memory observation | Direct NLP + memory |
| **5. Text Summarization** | BART summarization + memory observation | Direct NLP + memory |
| **6. Customer 360 View** | Unified view via agent routing | analytics_agent → MCP |
| **7. Sales Analytics** | Revenue by category via agent routing | analytics_agent → MCP |
| **8. Support Analytics** | Ticket metrics via agent routing | analytics_agent → MCP |

---

### Key Agentic Advantages over Nonagent Level-2

| Feature | Nonagent Level-2 | Analytics Agent |
|---|---|---|
| **Routing** | Fixed: always calls tool directly | Dynamic: Orchestrator classifies intent |
| **Tool Access** | Direct function import | MCP protocol (abstracted, auditable) |
| **Memory** | None | Agent observations recorded per analysis |
| **Audit** | Basic log | Full agent path + MCP call trace |
| **Autonomy** | Low — developer picks the tool | Medium — agent selects tool from MCP server |
| **Replanning** | None | Evaluation agent can trigger replan |

---

### Architecture Recap

```
agentic_ask("Segment customers...")
        │
        ▼
  Orchestrator Agent  →  intent = "analytics"
        │
        ▼
  Analytics Agent
        │
        ├── MCP: analytics_server.generate_segment(algorithm="rules")
        │         OR
        └── MCP: analytics_server.run_sql_query(sql=...)
                │
                ▼
        Memory: agent_observation.record_observation("analytics", ...)
                │
                ▼
        Response + Agent Path + MCP Calls + Audit Trail
```

---

### Technical Notes

**MCP Server**: `mcp_servers/analytics_server.py` — wraps `src/tools/analytics.py`

**Agent**: `agents/analytics_agent.py` — routes to generate_segment or run_sql_query

**Memory**: `memory/memory_manager.py` — agent_observation layer

**Data Sources**:
- `data/customers.db` — Customer profiles (SQLite)
- `data/sales_transactions.json` — Sales data
- `data/social_media.json` — Social media posts
- `data/support_tickets.json` — Support tickets

**Charts Saved To**: `data/charts/`

**Audit Log**: `data/audit.jsonl`
